In [1]:
%run "./Hamiltonian_solver.ipynb"

Best B: 4.1935340839189905
Transition frequency at B: 623.6441051016996
Error: 1.699504537100438e-09
[[         nan 458.98370144 461.92436004 464.86287142 467.79924027]
 [468.14934504 471.09215558 474.03281418 476.97132556 479.90769441]
 [480.60995082 483.55276136 486.49341996 489.43193134          nan]
 [         nan          nan 533.13126916 536.06978054 539.00614939]
 [         nan 536.90749676 539.84815536 542.78666674 545.72303559]
 [540.54867336 543.4914839  546.4321425  549.37065388 552.30702273]
 [546.94119669 549.88400724 552.82466584 555.76317721          nan]
 [553.04626773 555.98907828 558.92973688          nan          nan]
 [         nan          nan          nan 594.80710083 597.74346969]
 [         nan          nan 595.85182764 598.79033901 601.72670787]
 [         nan 597.25713907 600.19779767 603.13630905 606.0726779 ]
 [598.90607348 601.84888402 604.78954262 607.728054   610.66442285]
 [603.71987407 606.66268461 609.60334321 612.54185459          nan]
 [608.79568111 

In [2]:
import numpy as np
import matplotlib.pyplot as plt


S12_STATE_LABELS = [
    r"$|1,-1\rangle$",
    r"$|1,0\rangle$",
    r"$|1,+1\rangle$",
    r"$|2,+2\rangle$",
    r"$|2,+1\rangle$",
    r"$|2,0\rangle$",
    r"$|2,-1\rangle$",
    r"$|2,-2\rangle$",
]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ------------------------------------------------------------
# Fixed atomic and laser parameters
# ------------------------------------------------------------

B_gauss = 4.2079
wavelength_nm = 532.0


# ------------------------------------------------------------
# Widgets
# ------------------------------------------------------------

mode_widget = widgets.ToggleButtons(
    options=[
        ("Co-propagating", "co"),
        ("Counter-propagating", "counter"),
        ("Free beams", "free"),
    ],
    value="co",
    description="Geometry:",
    button_style="",
)


def angle_slider(description, value, minimum, maximum, step=1.0):
    return widgets.FloatSlider(
        description=description,
        value=value,
        min=minimum,
        max=maximum,
        step=step,
        continuous_update=False,
        readout=True,
        readout_format=".1f",
        style={"description_width": "90px"},
        layout=widgets.Layout(width="430px"),
    )


# Beam 1 controls
theta1_slider = angle_slider(
    r"Beam 1 θ:",
    value=90.0,
    minimum=0,
    maximum=180.0,
)

psi1_slider = angle_slider(
    r"Beam 1 ψ:",
    value=50.0,
    minimum=0.0,
    maximum=180.0,
)

chi1_slider = angle_slider(
    r"Beam 1 χ:",
    value=0.0,
    minimum=-45.0,
    maximum=45.0,
)

# Beam 2 controls
theta2_slider = angle_slider(
    r"Beam 2 θ:",
    value=90.0,
    minimum=0,
    maximum=180.0,
)

psi2_slider = angle_slider(
    r"Beam 2 ψ:",
    value=50.0,
    minimum=0.0,
    maximum=180.0,
)

chi2_slider = angle_slider(
    r"Beam 2 χ:",
    value=0.0,
    minimum=-45.0,
    maximum=45.0,
)


annotate_checkbox = widgets.Checkbox(
    value=True,
    description="Show numerical values",
    indent=False,
)

scientific_digits = widgets.IntSlider(
    value=3,
    min=1,
    max=6,
    step=1,
    description="Digits:",
    continuous_update=False,
    style={"description_width": "60px"},
    layout=widgets.Layout(width="260px"),
)


separate_colormaps_checkbox = widgets.Checkbox(
    value=True,
    description="Separate diagonal/off-diagonal color maps",
    indent=False,
)

normalize_checkbox = widgets.Checkbox(
    value=False,
    description="Normalize displayed values",
    indent=False,
)
status_label = widgets.HTML()

plot_output = widgets.Output()


# ------------------------------------------------------------
# Resolve the physical beam parameters from the selected mode
# ------------------------------------------------------------

def get_beam_angles():
    mode = mode_widget.value

    theta1 = float(theta1_slider.value)
    psi1 = float(psi1_slider.value)
    chi1 = float(chi1_slider.value)

    if mode == "co":
        # Identical directions and identical polarization.
        theta2 = theta1
        psi2 = psi1
        chi2 = chi1

    elif mode == "counter":
        # Supplementary propagation angles.
        # Polarizations remain independent.
        theta2 = 180.0 - theta1
        psi2 = float(psi2_slider.value)
        chi2 = float(chi2_slider.value)

    elif mode == "free":
        theta2 = float(theta2_slider.value)
        psi2 = float(psi2_slider.value)
        chi2 = float(chi2_slider.value)

    else:
        raise ValueError(f"Unknown geometry mode: {mode}")

    beam1_angles = np.array(
        [theta1, psi1, chi1],
        dtype=float,
    )

    beam2_angles = np.array(
        [theta2, psi2, chi2],
        dtype=float,
    )

    return beam1_angles, beam2_angles

In [8]:
def update_enabled_controls():
    mode = mode_widget.value

    if mode == "co":
        # Beam 2 is completely determined by beam 1.
        theta2_slider.disabled = True
        psi2_slider.disabled = True
        chi2_slider.disabled = True

        theta2_slider.value = theta1_slider.value
        psi2_slider.value = psi1_slider.value
        chi2_slider.value = chi1_slider.value

    elif mode == "counter":
        # theta_2 is constrained, but its polarization is independent.
        theta2_slider.disabled = True
        psi2_slider.disabled = False
        chi2_slider.disabled = False

        theta2_slider.value = 180.0 - theta1_slider.value

    elif mode == "free":
        theta2_slider.disabled = False
        psi2_slider.disabled = False
        chi2_slider.disabled = False

In [9]:
def recalculate_and_plot(change=None):
    update_enabled_controls()

    beam1_angles, beam2_angles = get_beam_angles()

    theta1, psi1, chi1 = beam1_angles
    theta2, psi2, chi2 = beam2_angles

    status_label.value = (
        "<b>Calculating Raman matrix...</b>"
    )

    try:
        beam1_polarization = (
            elliptical_polarization_from_geometry(
                beam_angle_deg=theta1,
                polarization_angle_deg=psi1,
                ellipticity_angle_deg=chi1,
            )
        )

        beam2_polarization = (
            elliptical_polarization_from_geometry(
                beam_angle_deg=theta2,
                polarization_angle_deg=psi2,
                ellipticity_angle_deg=chi2,
            )
        )

        raman = RamanTransitionStrength_S12(
            B_gauss=B_gauss,
            pol1=beam1_polarization,
            pol2=beam2_polarization,
            wavelength_nm=wavelength_nm,
        )

        # Raw unnormalized matrix.
        # The diagonal is retained.
        # ------------------------------------------------------------
        # Prepare displayed matrix and normalization
        # ------------------------------------------------------------

        raw_strength = np.asarray(
            raman["strength"],
            dtype=float,
        )
        maximum = float(np.max(raw_strength))
        number_of_states = raw_strength.shape[0]

        diagonal_mask = np.eye(
            number_of_states,
            dtype=bool,
        )

        off_diagonal_mask = ~diagonal_mask

        separate_colormaps = (
            separate_colormaps_checkbox.value
        )

        normalize_display = normalize_checkbox.value


        if separate_colormaps:
            # Keep diagonal and off-diagonal values in separate arrays.
            diagonal_values = np.where(
                diagonal_mask,
                raw_strength,
                np.nan,
            )

            off_diagonal_values = np.where(
                off_diagonal_mask,
                raw_strength,
                np.nan,
            )

            diagonal_maximum = np.nanmax(
                diagonal_values
            )

            off_diagonal_maximum = np.nanmax(
                off_diagonal_values
            )

            if normalize_display:
                if diagonal_maximum > 0:
                    diagonal_display = (
                        diagonal_values
                        / diagonal_maximum
                    )
                else:
                    diagonal_display = (
                        diagonal_values.copy()
                    )

                if off_diagonal_maximum > 0:
                    off_diagonal_display = (
                        off_diagonal_values
                        / off_diagonal_maximum
                    )
                else:
                    off_diagonal_display = (
                        off_diagonal_values.copy()
                    )
            else:
                diagonal_display = (
                    diagonal_values.copy()
                )

                off_diagonal_display = (
                    off_diagonal_values.copy()
                )

        else:
            whole_matrix_maximum = np.max(
                raw_strength
            )

            if (
                normalize_display
                and whole_matrix_maximum > 0
            ):
                combined_display = (
                    raw_strength
                    / whole_matrix_maximum
                )
            else:
                combined_display = (
                    raw_strength.copy()
                )

        with plot_output:
            clear_output(wait=True)

            fig, ax = plt.subplots(
                figsize=(11, 9)
            )

            if separate_colormaps:
                # Two transparent masked images are drawn on top
                # of each other.
                diagonal_image = ax.imshow(
                    np.ma.masked_invalid(
                        diagonal_display
                    ),
                    origin="upper",
                    interpolation="nearest",
                    aspect="equal",
                    cmap="plasma",
                )

                off_diagonal_image = ax.imshow(
                    np.ma.masked_invalid(
                        off_diagonal_display
                    ),
                    origin="upper",
                    interpolation="nearest",
                    aspect="equal",
                    cmap="viridis",
                )

            else:
                combined_image = ax.imshow(
                    combined_display,
                    origin="upper",
                    interpolation="nearest",
                    aspect="equal",
                    cmap="viridis",
                )

            ax.set_xticks(
                np.arange(number_of_states)
            )

            ax.set_yticks(
                np.arange(number_of_states)
            )

            ax.set_xticklabels(
                S12_STATE_LABELS,
                rotation=45,
                ha="right",
                rotation_mode="anchor",
            )

            ax.set_yticklabels(
                S12_STATE_LABELS
            )

            ax.set_xlabel(
                "Initial $6S_{1/2}$ state"
            )

            ax.set_ylabel(
                "Final $6S_{1/2}$ state"
            )

            geometry_names = {
                "co": "Co-propagating",
                "counter": "Counter-propagating",
                "free": "Independent beams",
            }

            normalization_text = (
                "normalized"
                if normalize_display
                else "unnormalized"
            )

            colormap_text = (
                "separate diagonal/off-diagonal scales"
                if separate_colormaps
                else "single shared scale"
            )

            ax.set_title(
                f"{normalization_text.capitalize()} Raman "
                "transition strengths\n"
                f"{geometry_names[mode_widget.value]}, "
                f"{colormap_text}\n"
                fr"Beam 1: "
                fr"$\theta_1={theta1:.1f}^\circ$, "
                fr"$\psi_1={psi1:.1f}^\circ$, "
                fr"$\chi_1={chi1:.1f}^\circ$"
                "\n"
                fr"Beam 2: "
                fr"$\theta_2={theta2:.1f}^\circ$, "
                fr"$\psi_2={psi2:.1f}^\circ$, "
                fr"$\chi_2={chi2:.1f}^\circ$"
            )

            # --------------------------------------------------------
            # Color bars
            # --------------------------------------------------------

            if separate_colormaps:
                diagonal_colorbar = fig.colorbar(
                    diagonal_image,
                    ax=ax,
                    fraction=0.046,
                    pad=0.07,
                )

                off_diagonal_colorbar = fig.colorbar(
                    off_diagonal_image,
                    ax=ax,
                    fraction=0.046,
                    pad=0.02,
                )

                if normalize_display:
                    diagonal_colorbar.set_label(
                        "Normalized diagonal strength"
                    )

                    off_diagonal_colorbar.set_label(
                        "Normalized off-diagonal strength"
                    )

                else:
                    diagonal_colorbar.set_label(
                        r"Diagonal strength "
                        r"$|\mathcal{A}_{ii}|^2$ "
                        r"$[(ea_0)^4/\mathrm{Hz}^2]$"
                    )

                    off_diagonal_colorbar.set_label(
                        r"Off-diagonal strength "
                        r"$|\mathcal{A}_{fi}|^2$ "
                        r"$[(ea_0)^4/\mathrm{Hz}^2]$"
                    )

            else:
                combined_colorbar = fig.colorbar(
                    combined_image,
                    ax=ax,
                    fraction=0.046,
                    pad=0.04,
                )

                if normalize_display:
                    combined_colorbar.set_label(
                        "Normalized strength"
                    )
                else:
                    combined_colorbar.set_label(
                        r"Unnormalized strength "
                        r"$|\mathcal{A}_{fi}|^2$ "
                        r"$[(ea_0)^4/\mathrm{Hz}^2]$"
                    )

            # --------------------------------------------------------
            # Cell boundaries
            # --------------------------------------------------------

            ax.set_xticks(
                np.arange(
                    -0.5,
                    number_of_states,
                    1.0,
                ),
                minor=True,
            )

            ax.set_yticks(
                np.arange(
                    -0.5,
                    number_of_states,
                    1.0,
                ),
                minor=True,
            )

            ax.grid(
                which="minor",
                linewidth=0.5,
            )

            ax.tick_params(
                which="minor",
                bottom=False,
                left=False,
            )

            # --------------------------------------------------------
            # Numerical annotations
            # --------------------------------------------------------

            if annotate_checkbox.value:
                digits = scientific_digits.value

                if separate_colormaps:
                    annotation_matrix = np.where(
                        diagonal_mask,
                        diagonal_display,
                        off_diagonal_display,
                    )
                else:
                    annotation_matrix = (
                        combined_display
                    )

                for final_state in range(
                    number_of_states
                ):
                    for initial_state in range(
                        number_of_states
                    ):
                        raw_value = raw_strength[
                            final_state,
                            initial_state,
                        ]

                        displayed_value = (
                            annotation_matrix[
                                final_state,
                                initial_state,
                            ]
                        )

                        if separate_colormaps:
                            if final_state == initial_state:
                                local_maximum = np.nanmax(
                                    diagonal_display
                                )
                            else:
                                local_maximum = np.nanmax(
                                    off_diagonal_display
                                )
                        else:
                            local_maximum = np.max(
                                combined_display
                            )

                        relative_value = (
                            displayed_value
                            / local_maximum
                            if local_maximum > 0
                            else 0.0
                        )

                        text_color = (
                            "black"
                            if relative_value > 0.45
                            else "white"
                        )

                        if normalize_display:
                            text = (
                                f"{displayed_value:.3f}"
                            )
                        else:
                            text = (
                                f"{raw_value:.{digits}e}"
                            )

                        ax.text(
                            initial_state,
                            final_state,
                            text,
                            ha="center",
                            va="center",
                            fontsize=10,
                            color=text_color,
                        )

            fig.tight_layout()
            plt.show()
            plt.close(fig)

        status_label.value = (
            f"<b>Finished.</b> "
            f"Maximum matrix element: "
            f"{maximum:.6e}"
        )

    except Exception as error:
        with plot_output:
            clear_output(wait=True)
            print(
                f"Calculation failed:\n"
                f"{type(error).__name__}: {error}"
            )

        status_label.value = (
            "<b style='color:red'>"
            "Calculation failed."
            "</b>"
        )

In [10]:
def parameter_changed(change):
    if change.get("name") != "value":
        return

    # Keep dependent controls visually synchronized.
    if mode_widget.value == "co":
        theta2_slider.value = (
            theta1_slider.value
        )
        psi2_slider.value = (
            psi1_slider.value
        )
        chi2_slider.value = (
            chi1_slider.value
        )

    elif mode_widget.value == "counter":
        theta2_slider.value = (
            180.0 - theta1_slider.value
        )

    recalculate_and_plot()


def mode_changed(change):
    if change.get("name") != "value":
        return

    update_enabled_controls()
    recalculate_and_plot()


# The physical sliders recalculate only after release because
# continuous_update=False was set above.
for slider in [
    theta1_slider,
    psi1_slider,
    chi1_slider,
    theta2_slider,
    psi2_slider,
    chi2_slider,
]:
    slider.observe(
        parameter_changed,
        names="value",
    )

mode_widget.observe(
    mode_changed,
    names="value",
)

annotate_checkbox.observe(
    recalculate_and_plot,
    names="value",
)

scientific_digits.observe(
    recalculate_and_plot,
    names="value",
)

separate_colormaps_checkbox.observe(
    recalculate_and_plot,
    names="value",
)

normalize_checkbox.observe(
    recalculate_and_plot,
    names="value",
)
90
beam1_box = widgets.VBox([
    widgets.HTML("<h4>Beam 1</h4>"),
    theta1_slider,
    psi1_slider,
    chi1_slider,
])

beam2_box = widgets.VBox([
    widgets.HTML("<h4>Beam 2</h4>"),
    theta2_slider,
    psi2_slider,
    chi2_slider,
])

display_options = widgets.VBox([
    widgets.HBox([
        annotate_checkbox,
        scientific_digits,
    ]),
    widgets.HBox([
        separate_colormaps_checkbox,
        normalize_checkbox,
    ]),
])

control_panel = widgets.VBox([
    mode_widget,
    widgets.HBox([
        beam1_box,
        beam2_box,
    ]),
    display_options,
    status_label,
])

app = widgets.VBox([
    control_panel,
    plot_output,
])

update_enabled_controls()
display(app)

# Initial calculation
recalculate_and_plot()

In [ ]:
b1 = 609.26034
b2 = 609.42916
c1 = 610.72612

In [ ]:
print(c1-b1)
print(c1-b2)

1.4657799999999952
1.2969600000000128


In [9]:
r1 = 612.02519
r2 = 612.19457
c1 = 610.72627

In [11]:
print(c1-r1)
print(c1-r2)

-1.2989199999999528
-1.4682999999999993


In [21]:

target_freq = 8037.75032 #in MHz
RamanPulseTime = 3000 #in us

def RepRateMode_CenterFreq(target_freq):
    rep_rate = 75.66255
    RepRateMode = int(target_freq/rep_rate)
    center_freq = target_freq - RepRateMode*rep_rate
    return RepRateMode, center_freq, rep_rate


rep_rate_stabilisation, centre_freq, rep_rate = RepRateMode_CenterFreq(target_freq)

In [22]:
print(rep_rate_stabilisation, centre_freq, rep_rate)

106 17.52002000000084 75.66255
